# MedBharat-1: Training Pipeline

**Architecture:** Curriculum-trained Mixture of Medical Experts with Dual-Script Support

**Novel contributions:**
1. Medical Sparse Attention (4 heads with learned medical entity bias)
2. Mixture of 6 Medical Experts (top-2 routing)
3. Curriculum training (learns like a medical student)
4. Dual-script tokenizer (English + Hindi)

**Total params:** ~500M | **Active params:** ~260M per forward pass

---

In [ ]:
# Cell 1: Install dependencies
!pip install -q tokenizers sentencepiece safetensors wandb datasets requests

In [ ]:
# Cell 2: Check GPU and set device
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    AMP_DTYPE = torch.float16
else:
    DEVICE = torch.device('cpu')
    print('WARNING: No GPU found! Training on CPU (will be very slow)')
    print('Make sure GPU is enabled: Kaggle Settings -> Accelerator -> GPU T4 x2')
    AMP_DTYPE = torch.bfloat16

print(f'\nUsing device: {DEVICE}')

In [ ]:
# Cell 3: Auto-download data if not present
import os
import json
import gzip
import subprocess
import sys
from pathlib import Path

OUTPUT_DIR = '/kaggle/working/medbharat-checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def count_data_files(d):
    if not Path(d).exists():
        return 0
    return sum(1 for f in Path(d).rglob('*') if f.is_file() and f.suffix in ('.gz', '.jsonl', '.txt'))

# Scan /kaggle/input/ to find our dataset
print('Scanning /kaggle/input/:')
input_dir = Path('/kaggle/input')
DATA_DIR = None

if input_dir.exists():
    for item in sorted(input_dir.iterdir()):
        n = count_data_files(str(item))
        print(f'  {item.name}/ -> {n} data files')
        if n >= 3 and DATA_DIR is None:
            DATA_DIR = str(item)

# Also check the expected path
for p in ['/kaggle/input/medbharat-healthcare-data']:
    if DATA_DIR is None and Path(p).exists():
        n = count_data_files(p)
        if n >= 3:
            DATA_DIR = p

if DATA_DIR and count_data_files(DATA_DIR) >= 3:
    print(f'\nUsing pre-attached data: {DATA_DIR} ({count_data_files(DATA_DIR)} files)')

    # If files are flat (prefixed with dir__), reorganize into subdirs for curriculum
    flat_files = [f for f in Path(DATA_DIR).iterdir() if f.is_file() and '__' in f.name]
    if flat_files:
        print('Reorganizing flat files into subdirectories...')
        REORG_DIR = '/kaggle/working/data'
        os.makedirs(REORG_DIR, exist_ok=True)
        import shutil
        for f in Path(DATA_DIR).iterdir():
            if not f.is_file() or f.suffix not in ('.gz', '.jsonl', '.txt'):
                continue
            if '__' in f.name:
                subdir, fname = f.name.split('__', 1)
            else:
                subdir, fname = 'misc', f.name
            dest = Path(REORG_DIR) / subdir
            dest.mkdir(exist_ok=True)
            shutil.copy2(f, dest / fname)
        DATA_DIR = REORG_DIR
        print(f'Reorganized into {REORG_DIR} ({count_data_files(REORG_DIR)} files)')
else:
    print('\nNo pre-attached data found. Auto-downloading from HuggingFace...')
    DATA_DIR = '/kaggle/working/data'
    os.makedirs(DATA_DIR, exist_ok=True)

    from datasets import load_dataset

    dirs = ['wiki_medical','ncbi_books','pubmed','icd11','nlem_2025',
            'drugbank','medqa','pubmedqa','who_india','medmcqa']
    for d in dirs:
        os.makedirs(f'{DATA_DIR}/{d}', exist_ok=True)

    # 1. MedMCQA — 194K Indian medical MCQs (most critical)
    out = Path(DATA_DIR) / 'medmcqa' / 'medmcqa_train.jsonl.gz'
    if not out.exists():
        print('  Downloading MedMCQA (194K Indian medical MCQs)...')
        try:
            ds = load_dataset('openlifescienceai/medmcqa', split='train')
            with gzip.open(out, 'wt', encoding='utf-8') as f:
                for row in ds:
                    rec = {'question': row.get('question',''),
                           'options': [row.get('opa',''), row.get('opb',''),
                                       row.get('opc',''), row.get('opd','')],
                           'answer': row.get('cop', 0),
                           'explanation': row.get('exp',''),
                           'subject': row.get('subject_name','')}
                    f.write(json.dumps(rec, ensure_ascii=False) + '\n')
            print(f'    medmcqa: {len(ds)} records')
        except Exception as e:
            print(f'    WARN: MedMCQA download failed: {e}')

    # 2. Wiki Medical Terms
    out = Path(DATA_DIR) / 'wiki_medical' / 'wiki_medical_terms.jsonl.gz'
    if not out.exists():
        print('  Downloading Wiki Medical Terms...')
        try:
            ds = load_dataset('gamino/wiki_medical_terms', split='train')
            with gzip.open(out, 'wt', encoding='utf-8') as f:
                for row in ds:
                    text = row.get('page_text','') or row.get('text','')
                    title = row.get('page_title','') or row.get('title','')
                    if text and len(text) > 50:
                        f.write(json.dumps({'title': title, 'text': text}, ensure_ascii=False) + '\n')
            print(f'    wiki_medical: {len(ds)} records')
        except Exception as e:
            print(f'    WARN: Wiki Medical download failed: {e}')

    # 3. MedQA USMLE
    out = Path(DATA_DIR) / 'medqa' / 'medqa_usmle.jsonl.gz'
    if not out.exists():
        print('  Downloading MedQA...')
        try:
            ds = load_dataset('GBaker/MedQA-USMLE-4-options', split='train')
            with gzip.open(out, 'wt', encoding='utf-8') as f:
                for row in ds:
                    opts = row.get('options', {})
                    if isinstance(opts, dict):
                        opts = list(opts.values())
                    f.write(json.dumps({'question': row.get('question',''),
                                        'answer': row.get('answer',''),
                                        'options': opts}, ensure_ascii=False) + '\n')
            print(f'    medqa: {len(ds)} records')
        except Exception as e:
            print(f'    WARN: MedQA download failed: {e}')

    # 4. PubMedQA
    out = Path(DATA_DIR) / 'pubmedqa' / 'pubmedqa_labeled.jsonl.gz'
    if not out.exists():
        print('  Downloading PubMedQA...')
        try:
            ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')
            with gzip.open(out, 'wt', encoding='utf-8') as f:
                for row in ds:
                    ctx = row.get('context', {})
                    ctx_text = ' '.join(ctx.get('contexts',[])) if isinstance(ctx, dict) else str(ctx)
                    f.write(json.dumps({'question': row.get('question',''),
                                        'long_answer': row.get('long_answer',''),
                                        'context': ctx_text}, ensure_ascii=False) + '\n')
            print(f'    pubmedqa: {len(ds)} records')
        except Exception as e:
            print(f'    WARN: PubMedQA download failed: {e}')

    n_files = count_data_files(DATA_DIR)
    print(f'\nAuto-download complete. {n_files} files in {DATA_DIR}')

# List data files
print(f'\nData directory: {DATA_DIR}')
for f in sorted(Path(DATA_DIR).rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / (1024*1024)
        print(f'  {f.relative_to(DATA_DIR)} ({size_mb:.1f} MB)')
print(f'Total: {count_data_files(DATA_DIR)} files')

In [ ]:
# Cell 4: Model Architecture (all in one cell for Kaggle)
import math
import json
import gzip
import gc
import time
import random
from dataclasses import dataclass
from typing import List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from torch.amp import GradScaler, autocast


# ===== CONFIG =====
@dataclass
class MedBharatConfig:
    vocab_size: int = 48_000
    max_seq_len: int = 2048
    d_model: int = 1024
    pad_token_id: int = 0
    bos_token_id: int = 1
    eos_token_id: int = 2
    n_shared_layers: int = 8
    n_heads: int = 16
    n_medical_heads: int = 4
    head_dim: int = 64
    intermediate_size: int = 4096
    n_experts: int = 6
    n_expert_layers: int = 8
    top_k_experts: int = 2
    expert_intermediate_size: int = 2048
    router_aux_loss_weight: float = 0.01
    n_languages: int = 3
    rms_norm_eps: float = 1e-6
    rope_theta: float = 10000.0
    dropout: float = 0.0
    attention_dropout: float = 0.0
    gradient_checkpointing: bool = True
    expert_names: tuple = ('diagnosis', 'pharmacology', 'treatment', 'ayush', 'anatomy', 'general')

config = MedBharatConfig()
print('Config loaded')

In [ ]:
# Cell 5: Core Layers

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).to(x.dtype) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)
    def forward(self, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, intermediate_size, bias=False)
        self.up_proj = nn.Linear(d_model, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, d_model, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

print('Core layers defined')

In [ ]:
# Cell 6: Medical Sparse Attention + Transformer Block

class MedicalSparseAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.n_medical_heads = cfg.n_medical_heads
        self.head_dim = cfg.head_dim
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * cfg.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_heads * cfg.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_heads * cfg.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.medical_gate = nn.Linear(cfg.d_model, cfg.n_medical_heads, bias=True)
        self.rotary_emb = RotaryEmbedding(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)

    def forward(self, hidden_states, attention_mask=None):
        B, S, _ = hidden_states.shape
        q = self.q_proj(hidden_states).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.rotary_emb(S)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.full((S, S), float('-inf'), device=hidden_states.device), diagonal=1)
        attn = attn + causal.unsqueeze(0).unsqueeze(0)
        if attention_mask is not None:
            attn = attn.masked_fill(attention_mask.unsqueeze(1).unsqueeze(2) == 0, float('-inf'))
        # Medical bias for last n_medical_heads
        med_bias = torch.sigmoid(self.medical_gate(hidden_states)).permute(0, 2, 1).unsqueeze(2)
        n_std = self.n_heads - self.n_medical_heads
        attn = torch.cat([attn[:, :n_std], attn[:, n_std:] + med_bias], dim=1)
        attn = F.softmax(attn, dim=-1, dtype=torch.float32).to(q.dtype)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, S, -1)
        return self.o_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, cfg, intermediate_size=None):
        super().__init__()
        inter = intermediate_size or cfg.intermediate_size
        self.attention_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.attention = MedicalSparseAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.ffn = SwiGLUFFN(cfg.d_model, inter)
    def forward(self, x, attention_mask=None):
        x = x + self.attention(self.attention_norm(x), attention_mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x

print('Attention + TransformerBlock defined')

In [ ]:
# Cell 7: Mixture of Medical Experts

class ExpertRouter(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.n_experts, bias=False)
        self.top_k = cfg.top_k_experts
        self.n_experts = cfg.n_experts
        self.aux_loss_weight = cfg.router_aux_loss_weight

    def forward(self, hidden_states):
        logits = self.gate(hidden_states)
        weights, indices = torch.topk(logits, self.top_k, dim=-1)
        weights = F.softmax(weights, dim=-1, dtype=torch.float32).to(hidden_states.dtype)
        probs = F.softmax(logits, dim=-1, dtype=torch.float32)
        tokens_per_expert = probs.mean(dim=[0, 1])
        aux_loss = self.aux_loss_weight * self.n_experts * (tokens_per_expert * tokens_per_expert).sum()
        return weights, indices, aux_loss


class ExpertModule(nn.Module):
    def __init__(self, cfg, name=''):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(cfg, intermediate_size=cfg.expert_intermediate_size)
            for _ in range(cfg.n_expert_layers)
        ])
    def forward(self, x, attention_mask=None):
        for layer in self.layers:
            x = layer(x, attention_mask)
        return x


class MixtureOfMedicalExperts(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.router = ExpertRouter(cfg)
        self.experts = nn.ModuleList([
            ExpertModule(cfg, name=n) for n in cfg.expert_names
        ])

    def forward(self, hidden_states, attention_mask=None):
        weights, indices, aux_loss = self.router(hidden_states)
        final = torch.zeros_like(hidden_states)
        for eidx in range(self.cfg.n_experts):
            mask = (indices == eidx)
            if not mask.any():
                continue
            expert_out = self.experts[eidx](hidden_states, attention_mask)
            for k in range(self.cfg.top_k_experts):
                slot_mask = mask[:, :, k].unsqueeze(-1)
                slot_weight = weights[:, :, k].unsqueeze(-1)
                final = final + expert_out * slot_mask * slot_weight
        return final, aux_loss

print('MoE layer defined')

In [ ]:
# Cell 8: Full MedBharat-1 Model

class MedBharatModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.language_embedding = nn.Embedding(cfg.n_languages, cfg.d_model)
        self.shared_layers = nn.ModuleList([
            TransformerBlock(cfg) for _ in range(cfg.n_shared_layers)
        ])
        self.shared_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.moe = MixtureOfMedicalExperts(cfg)
        self.moe_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.output_norm = RMSNorm(cfg.d_model, cfg.rms_norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # weight tying
        self.apply(self._init_weights)
        self._gc = cfg.gradient_checkpointing

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, language_ids=None, labels=None):
        h = self.token_embedding(input_ids)
        if language_ids is not None:
            h = h + self.language_embedding(language_ids)
        for layer in self.shared_layers:
            if self._gc and self.training:
                h = torch.utils.checkpoint.checkpoint(layer, h, attention_mask, use_reentrant=False)
            else:
                h = layer(h, attention_mask)
        h = self.shared_norm(h)
        if self._gc and self.training:
            def _moe_forward(hidden, mask):
                return self.moe(hidden, mask)
            moe_out, aux_loss = torch.utils.checkpoint.checkpoint(
                _moe_forward, h, attention_mask, use_reentrant=False)
        else:
            moe_out, aux_loss = self.moe(h, attention_mask)
        h = h + self.moe_norm(moe_out)
        h = self.output_norm(h)
        logits = self.lm_head(h)
        loss = None
        total_loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1, :].contiguous().view(-1, self.cfg.vocab_size),
                labels[:, 1:].contiguous().view(-1),
                ignore_index=self.cfg.pad_token_id)
            total_loss = loss + aux_loss
        return {'logits': logits, 'loss': loss, 'aux_loss': aux_loss, 'total_loss': total_loss}


# Build model
model = MedBharatModel(config)
total_params = sum(p.numel() for p in model.parameters())
print(f'MedBharat-1 built: {total_params/1e6:.1f}M total parameters')
model = model.to(DEVICE)
print(f'Model on {DEVICE}')

In [ ]:
# Cell 9: Tokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors, decoders
from tokenizers.normalizers import Sequence, NFKC

TOKENIZER_PATH = f'{OUTPUT_DIR}/medbharat_tokenizer.json'

def train_tokenizer(corpus_files, vocab_size=48000):
    tok = Tokenizer(models.BPE(unk_token='<unk>'))
    tok.normalizer = Sequence([NFKC()])
    tok.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.WhitespaceSplit(),
        pre_tokenizers.Punctuation(),
    ])
    special = ['<pad>', '<s>', '</s>', '<unk>', '<mask>',
               '<en>', '<hi>', '<mix>', '<question>', '<answer>',
               '<drug>', '<disease>', '<treatment>']
    trainer_obj = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=3,
        special_tokens=special, show_progress=True,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet())

    def text_iter():
        for fp in corpus_files:
            p = Path(fp)
            try:
                opener = gzip.open if p.suffix == '.gz' else open
                with opener(p, 'rt', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if not line or len(line) < 20:
                            continue
                        if line.startswith('{'):
                            try:
                                rec = json.loads(line)
                                for field in ['text','title','question','answer','explanation']:
                                    v = rec.get(field, '')
                                    if v and len(v) > 10:
                                        yield v
                            except: pass
                        else:
                            yield line
            except: continue

    tok.train_from_iterator(text_iter(), trainer=trainer_obj)
    tok.post_processor = processors.TemplateProcessing(
        single='<s> $A </s>', pair='<s> $A </s> <s> $B </s>',
        special_tokens=[('<s>', tok.token_to_id('<s>')), ('</s>', tok.token_to_id('</s>'))])
    tok.decoder = decoders.BPEDecoder()
    tok.save(TOKENIZER_PATH)
    print(f'Tokenizer trained: {tok.get_vocab_size()} tokens -> {TOKENIZER_PATH}')
    return tok

# Train or load tokenizer
if Path(TOKENIZER_PATH).exists():
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
    print(f'Loaded tokenizer: {tokenizer.get_vocab_size()} tokens')
else:
    corpus_files = [str(f) for f in Path(DATA_DIR).rglob('*')
                    if f.suffix in ('.gz', '.jsonl', '.txt') and f.stat().st_size > 100]
    print(f'Training tokenizer on {len(corpus_files)} files...')
    tokenizer = train_tokenizer(corpus_files)

In [ ]:
# Cell 10: Dataset + Curriculum

def detect_language(text):
    hindi = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    alpha = sum(1 for c in text if c.isalpha())
    if alpha == 0: return 0
    r = hindi / alpha
    return 1 if r > 0.5 else (2 if r > 0.1 else 0)


class MedBharatDataset(IterableDataset):
    def __init__(self, data_files, tokenizer, max_seq_len=2048):
        self.data_files = data_files
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def _read_texts(self):
        files = list(self.data_files)
        random.shuffle(files)
        for fp in files:
            p = Path(fp)
            try:
                opener = gzip.open if p.suffix == '.gz' else open
                with opener(p, 'rt', encoding='utf-8') as f:
                    buf = []
                    for line in f:
                        line = line.strip()
                        if not line:
                            if buf:
                                yield ' '.join(buf)
                                buf = []
                            continue
                        if line.startswith('{'):
                            try:
                                rec = json.loads(line)
                                parts = [rec.get(f,'') for f in ['title','text','question','answer','explanation','long_answer','context'] if rec.get(f)]
                                if parts:
                                    yield ' '.join(parts)
                            except: pass
                        else:
                            buf.append(line)
                    if buf:
                        yield ' '.join(buf)
            except: continue

    def __iter__(self):
        tok_buf, lang_buf = [], []
        for text in self._read_texts():
            if len(text) < 50: continue
            enc = self.tokenizer.encode(text)
            lang = detect_language(text)
            tok_buf.extend(enc.ids)
            lang_buf.extend([lang] * len(enc.ids))
            while len(tok_buf) >= self.max_seq_len + 1:
                yield {
                    'input_ids': torch.tensor(tok_buf[:self.max_seq_len], dtype=torch.long),
                    'labels': torch.tensor(tok_buf[1:self.max_seq_len+1], dtype=torch.long),
                    'language_ids': torch.tensor(lang_buf[:self.max_seq_len], dtype=torch.long),
                }
                tok_buf = tok_buf[self.max_seq_len:]
                lang_buf = lang_buf[self.max_seq_len:]


# Curriculum phases
PHASES = {
    1: {'name': 'Pre-Med', 'keywords': ['wiki', 'book', 'anatomy', 'physiology'], 'epochs': 2},
    2: {'name': 'Pathology', 'keywords': ['pubmed', 'icd'], 'epochs': 2},
    3: {'name': 'Pharmacology', 'keywords': ['nlem', 'drug'], 'epochs': 3},
    4: {'name': 'Clinical', 'keywords': ['medqa', 'pubmedqa', 'who'], 'epochs': 2},
    5: {'name': 'Indian Spec.', 'keywords': ['medmcqa', 'ayush', 'india'], 'epochs': 3},
    6: {'name': 'Residency', 'keywords': None, 'epochs': 1},  # ALL data
}

def get_phase_files(phase):
    # Include .jsonl files too (Kaggle auto-decompresses .gz -> .jsonl)
    all_files = [str(f) for f in Path(DATA_DIR).rglob('*')
                 if f.is_file() and f.suffix in ('.gz', '.jsonl', '.txt') and f.stat().st_size > 100]
    if PHASES[phase]['keywords'] is None:
        return all_files  # Phase 6: everything
    keywords = PHASES[phase]['keywords']
    return [f for f in all_files if any(k in f.lower() for k in keywords)] or all_files

print(f'Dataset and Curriculum ready. {len(PHASES)} phases defined.')
print(f'All data files:')
for f in get_phase_files(6):
    print(f'  {Path(f).name} ({Path(f).stat().st_size/1024:.0f} KB)')

In [ ]:
# Cell 11: Training Loop

# Hyperparameters
BATCH_SIZE = 4          # fits P100 16GB with grad checkpointing
GRAD_ACCUM = 8          # effective batch = 32
PEAK_LR = 3e-4
MIN_LR = 1e-5
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0
LOG_EVERY = 25
SAVE_EVERY = 250

optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, betas=(0.9, 0.95), weight_decay=0.1)

USE_AMP = DEVICE.type == 'cuda'
scaler = GradScaler("cuda") if USE_AMP else None

global_step = 0
tokens_seen = 0
best_loss = float('inf')

# Resume from checkpoint if exists
ckpt_path = Path(OUTPUT_DIR) / 'latest_checkpoint'
if ckpt_path.exists():
    print('Resuming from checkpoint...')
    model.load_state_dict(torch.load(ckpt_path / 'model.pt', map_location=DEVICE, weights_only=True))
    optimizer.load_state_dict(torch.load(ckpt_path / 'optimizer.pt', map_location=DEVICE, weights_only=True))
    if scaler is not None and (ckpt_path / 'scaler.pt').exists():
        scaler.load_state_dict(torch.load(ckpt_path / 'scaler.pt', map_location=DEVICE, weights_only=True))
    with open(ckpt_path / 'state.json') as f:
        state = json.load(f)
    global_step = state['global_step']
    tokens_seen = state['tokens_seen']
    best_loss = state['best_loss']
    START_PHASE = state['current_phase']
    print(f'Resumed: step={global_step}, phase={START_PHASE}')
else:
    START_PHASE = 1

TOTAL_STEPS = 50000  # estimate

def get_lr():
    if global_step < WARMUP_STEPS:
        return PEAK_LR * global_step / max(1, WARMUP_STEPS)
    progress = min(1.0, (global_step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS))
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1.0 + math.cos(math.pi * progress))

def save_checkpoint(name='latest_checkpoint', phase=1):
    d = Path(OUTPUT_DIR) / name
    d.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), d / 'model.pt')
    torch.save(optimizer.state_dict(), d / 'optimizer.pt')
    if scaler is not None:
        torch.save(scaler.state_dict(), d / 'scaler.pt')
    with open(d / 'state.json', 'w') as f:
        json.dump({'global_step': global_step, 'tokens_seen': tokens_seen,
                    'best_loss': best_loss, 'current_phase': phase}, f)
    print(f'  Saved checkpoint: {name} (step {global_step})')

print(f'Training setup ready. Device: {DEVICE}, AMP: {USE_AMP}')
print(f'Starting from phase {START_PHASE}, step {global_step}')

In [ ]:
# Cell 12: RUN TRAINING

print('='*60)
print('  MEDBHARAT-1 CURRICULUM TRAINING')
print(f'  Device: {DEVICE} | AMP: {USE_AMP}')
print('='*60)

for phase in range(START_PHASE, 7):
    phase_cfg = PHASES[phase]
    files = get_phase_files(phase)
    print(f'\n--- Phase {phase}: {phase_cfg["name"]} ({len(files)} files, {phase_cfg["epochs"]} epochs) ---')

    if not files:
        print(f'  No files for phase {phase}, skipping')
        continue

    for epoch in range(phase_cfg['epochs']):
        dataset = MedBharatDataset(files, tokenizer, config.max_seq_len)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0, pin_memory=(DEVICE.type=='cuda'))
        model.train()
        optimizer.zero_grad()
        epoch_loss = 0.0
        epoch_steps = 0
        t0 = time.time()

        for batch_idx, batch in enumerate(loader):
            ids = batch['input_ids'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            lang = batch['language_ids'].to(DEVICE)

            if USE_AMP:
                with autocast("cuda", dtype=AMP_DTYPE):
                    out = model(input_ids=ids, language_ids=lang, labels=labels)
                    loss = out['total_loss'] / GRAD_ACCUM
                scaler.scale(loss).backward()
            else:
                out = model(input_ids=ids, language_ids=lang, labels=labels)
                loss = out['total_loss'] / GRAD_ACCUM
                loss.backward()

            if (batch_idx + 1) % GRAD_ACCUM == 0:
                if USE_AMP:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    lr = get_lr()
                    for pg in optimizer.param_groups: pg['lr'] = lr
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    lr = get_lr()
                    for pg in optimizer.param_groups: pg['lr'] = lr
                    optimizer.step()

                optimizer.zero_grad()
                global_step += 1
                tokens_seen += BATCH_SIZE * GRAD_ACCUM * config.max_seq_len
                actual_loss = loss.item() * GRAD_ACCUM
                epoch_loss += actual_loss
                epoch_steps += 1

                if global_step % LOG_EVERY == 0:
                    avg = epoch_loss / epoch_steps
                    elapsed = time.time() - t0
                    tps = tokens_seen / elapsed if elapsed > 0 else 0
                    ppl = math.exp(min(avg, 20))
                    print(f'  P{phase} E{epoch+1} Step {global_step} | Loss {actual_loss:.4f} | '
                          f'Avg {avg:.4f} | PPL {ppl:.1f} | LR {lr:.2e} | Tok {tokens_seen/1e6:.1f}M | '
                          f'Aux {out["aux_loss"].item():.4f}')

                if global_step % SAVE_EVERY == 0:
                    save_checkpoint('latest_checkpoint', phase)
                    if actual_loss < best_loss:
                        best_loss = actual_loss
                        save_checkpoint('best_model', phase)

            del ids, labels, lang, out, loss
            if batch_idx % 50 == 0:
                gc.collect()
                if DEVICE.type == 'cuda':
                    torch.cuda.empty_cache()

        if epoch_steps > 0:
            print(f'  Phase {phase} Epoch {epoch+1} done | Avg Loss: {epoch_loss/epoch_steps:.4f}')
        save_checkpoint(f'phase{phase}_epoch{epoch+1}', phase)

save_checkpoint('final_model', 6)
print(f'\nTraining complete! Steps: {global_step} | Tokens: {tokens_seen/1e9:.2f}B | Best loss: {best_loss:.4f}')

In [ ]:
# Cell 13: Benchmark Evaluation

BENCHMARK = [
    {'q': 'What is the schedule classification of Paracetamol in NLEM?', 'opts': ['Schedule H', 'Schedule X', 'OTC', 'Schedule H1'], 'ans': 2, 'cat': 'indian_drugs'},
    {'q': 'DOTS regimen for TB includes all EXCEPT:', 'opts': ['Isoniazid', 'Rifampicin', 'Azithromycin', 'Pyrazinamide'], 'ans': 2, 'cat': 'indian_drugs'},
    {'q': 'Most common serotype causing severe dengue in India:', 'opts': ['DEN-1', 'DEN-2', 'DEN-3', 'DEN-4'], 'ans': 1, 'cat': 'tropical_disease'},
    {'q': 'Kala-azar in India is most prevalent in:', 'opts': ['Kerala', 'Bihar', 'Tamil Nadu', 'Punjab'], 'ans': 1, 'cat': 'tropical_disease'},
    {'q': 'The three doshas in Ayurveda are:', 'opts': ['Vata, Pitta, Kapha', 'Sattva, Rajas, Tamas', 'Rasa, Rakta, Mamsa', 'Agni, Soma, Vayu'], 'ans': 0, 'cat': 'ayush'},
    {'q': 'ASHA workers are part of which mission?', 'opts': ['NRHM/NHM', 'PMJAY', 'ICDS', 'RSBY'], 'ans': 0, 'cat': 'indian_exam'},
    {'q': 'First-line treatment for DKA:', 'opts': ['Oral metformin', 'IV insulin + fluids + K+', 'Sulfonylurea', 'Diet control'], 'ans': 1, 'cat': 'clinical'},
    {'q': 'INH prophylaxis supplement:', 'opts': ['Vitamin C', 'Pyridoxine (B6)', 'Vitamin D', 'Iron'], 'ans': 1, 'cat': 'clinical'},
    {'q': 'Dengue ka sabse aam lakshan? (Most common dengue symptom?)', 'opts': ['Khansi', 'Tez bukhar + joint pain', 'Pet dard', 'Sujan'], 'ans': 1, 'cat': 'hindi'},
    {'q': 'Pulse Polio uses which vaccine?', 'opts': ['IPV', 'OPV', 'BCG', 'MMR'], 'ans': 1, 'cat': 'indian_exam'},
]

model.eval()
correct = 0
results_by_cat = {}

for item in BENCHMARK:
    scores = []
    for opt in item['opts']:
        prompt = f"Medical Question: {item['q']}\nAnswer: {opt}"
        enc = tokenizer.encode(prompt)
        ids = torch.tensor([enc.ids], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            logits = model(ids)['logits']
        # Score = mean log prob of last few tokens
        lp = F.log_softmax(logits[0, -5:-1], dim=-1)
        score = lp.gather(1, ids[0, -4:].unsqueeze(1)).mean().item()
        scores.append(score)
    
    pred = scores.index(max(scores))
    is_correct = pred == item['ans']
    correct += is_correct
    cat = item['cat']
    results_by_cat.setdefault(cat, []).append(is_correct)
    status = 'CORRECT' if is_correct else 'WRONG'
    print(f'  [{status}] {item["q"][:60]}... -> {item["opts"][pred]} (expected: {item["opts"][item["ans"]]})')

print(f'\n{"="*50}')
print(f'OVERALL: {correct}/{len(BENCHMARK)} = {correct/len(BENCHMARK):.1%}')
for cat, results in results_by_cat.items():
    print(f'  {cat}: {sum(results)}/{len(results)} = {sum(results)/len(results):.1%}')
print(f'{"="*50}')

In [ ]:
# Cell 14: Test Generation

@torch.no_grad()
def generate(prompt, max_tokens=200, temperature=0.8):
    model.eval()
    enc = tokenizer.encode(prompt)
    ids = torch.tensor([enc.ids], dtype=torch.long).to(DEVICE)
    for _ in range(max_tokens):
        logits = model(ids[:, -config.max_seq_len:])['logits'][:, -1, :]
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)
        ids = torch.cat([ids, nxt], dim=1)
        if nxt.item() == config.eos_token_id:
            break
    return tokenizer.decode(ids[0].tolist())

# Test prompts
prompts = [
    'The first-line treatment for tuberculosis in India is',
    'Dengue fever is caused by',
    'In Ayurveda, the three doshas are',
    'Paracetamol is classified as',
]

for p in prompts:
    print(f'\nPrompt: {p}')
    print(f'Output: {generate(p, max_tokens=100)}')
    print('-'*50)

In [ ]:
# Cell 15: Save final model for download
import shutil

final_dir = Path(OUTPUT_DIR) / 'medbharat-1-final'
final_dir.mkdir(exist_ok=True)

# Save model
torch.save(model.state_dict(), final_dir / 'model.pt')

# Save config
with open(final_dir / 'config.json', 'w') as f:
    json.dump(vars(config), f, indent=2, default=str)

# Copy tokenizer
shutil.copy2(TOKENIZER_PATH, final_dir / 'tokenizer.json')

# Save benchmark results
with open(final_dir / 'benchmark.json', 'w') as f:
    json.dump({
        'overall_accuracy': correct / len(BENCHMARK),
        'total_params': total_params,
        'tokens_trained': tokens_seen,
        'categories': {cat: sum(r)/len(r) for cat, r in results_by_cat.items()}
    }, f, indent=2)

print(f'Final model saved to {final_dir}')
print(f'Files: {[f.name for f in final_dir.iterdir()]}')
print(f'\nDownload from Kaggle Output tab or upload to HuggingFace!')